Adapted from https://github.com/dargueso/EHF/

In [1]:
import netCDF4 as nc
import xarray as xr
import numpy as np
import glob as glob
import pandas as pd
import os
import pdb
from itertools import groupby
from constants import const
from pathlib import Path
import datetime as dt

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

/g/data/ng72/ms5578/ID_HW_BARRA


In [3]:
client = Client()
client

2025-04-16 14:53:35,436 - distributed.preloading - INFO - Creating preload: /g/data/hh5/public/apps/dask-optimiser/schedplugin.py
2025-04-16 14:53:35,439 - distributed.utils - INFO - Reload module schedplugin from .py file
2025-04-16 14:53:35,443 - distributed.preloading - INFO - Import preload module: /g/data/hh5/public/apps/dask-optimiser/schedplugin.py


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 7
Total threads: 28,Total memory: 125.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39175,Workers: 7
Dashboard: http://127.0.0.1:8787/status,Total threads: 28
Started: Just now,Total memory: 125.19 GiB
Comm: tcp://127.0.0.1:45435,Total threads: 4
Dashboard: http://127.0.0.1:33481/status,Memory: 17.88 GiB
Nanny: tcp://127.0.0.1:38601,


In [4]:
def calc_percentile(tave, nyears, thres_file=None, method="NF13", nwindow=15):

    """Function to calculate the percentile that indentifies hot days
    tave: mean daily temperature calcualted from tmax and tmin
    nyears: number of years in the analysed period
    thres_file: file that contains previously calculated percentiles
    method: now two methods are supported depending on how the percentiles are calculated 'NF13' and 'PA13'
    nwindow: number of days for the window used to calculate percentiles in PA13 method
    ---
    output: pct_calc
    """
    if method == "NF13":

        if thres_file == None:
            print("No thresholds file provided, we will calculate them")

            if not isinstance(tave, np.ma.core.MaskedArray):
                pct_calc = np.nanpercentile(tave, 95, axis=0)

            else:
                pct_calc = np.ones(tave.shape[1:], float) * const.missingval
                for i in range(tave.shape[1]):
                    for j in range(tave.shape[2]):
                        aux = tave[:, i, j]
                        if len(aux[~aux.mask].data) != 0:
                            pct_calc[i, j] = np.nanpercentile(
                                aux[~aux.mask].data, 95, axis=0
                            )
        else:
            print("Percentiles are retrieved from the thfile provided")
            pct_file = nc.Dataset(thres_file, "r")
            pct_calc = pct_file.variables["PRCTILE95"][:].astype("float")

    elif method == "PA13":

        if thres_file == None:
            # No percentile file is provided and thus they are calculated from the given data
            print("Percentiles are calculated because no thfile is provided")
            windowrange = np.zeros((365,), dtype=bool)
            windowrange[: int(np.ceil(nwindow / 2))] = True
            windowrange[-int(np.floor(nwindow / 2)) :] = True
            if np.sum(windowrange) != nwindow:
                raise SystemExit(0)
            windowrange = np.tile(windowrange, nyears)
            pct_calc = np.ones((365,) + tave.shape[1:], float) * const.missingval

            if not isinstance(tave, np.ma.core.MaskedArray):
                for d in range(365):
                    pct_calc[d, :, :] = np.percentile(
                        tave[windowrange == True, :, :], 90, axis=0
                    )
                    windowrange = np.roll(windowrange, 1)

            else:
                for i in range(tave.shape[1]):
                    for j in range(tave.shape[2]):
                        for d in range(365):
                            aux = tave[windowrange == True, :, :]
                            if len(aux[~aux.mask].data) != 0:
                                pct_calc[d, :, :] = np.percentile(
                                    aux[~aux.mask].data, 90, axis=0
                                )
                            windowrange = np.roll(windowrange, 1)

        else:
            print("Percentiles are retrieved from the thfile provided")
            # A percentile file is provided and it contains a PRCTILE90 variable
            pct_file = nc.Dataset(thres_file, "r")
            pct_calc = pct_file.variables["PRCTILE90"][:].astype("float")

    else:
        raise ValueError("Method not supported: Choose between NF13 or PA13")

    return pct_calc


In [5]:
def calc_spell(series):

    if isinstance(series, np.ma.core.MaskedArray):
        if np.any(series.mask == True):
            series[series.mask] = -99

    srun = np.zeros(series.shape)
    srun[1:] = np.diff(series, axis=0)
    srun[srun == 99] = -1
    srun[srun == 100] = 1
    srun[0] = -1
    if isinstance(series, np.ma.core.MaskedArray):
        L = (series.data).tolist()
    else:
        L = (series).tolist()
    groups_hw = []

    for k, g in groupby(L):
        if k == 1:
            b = list(g)
            groups_hw.append(sum(b))

    spell_hw = np.zeros((len(series),), dtype=int)
    if np.any(srun == 1):
        spell_hw[srun == 1] = np.asarray(groups_hw)

    ## Keep only spells equal or larger than 3 days

    spell_hw[spell_hw < 3] = 0
    return spell_hw

In [6]:
syear = 2009
eyear = 2010

fin = xr.open_dataset("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/tas_hwperiod_short.nc")

tave = fin.tas.chunk(time=-1,lat=-1,lon="350Mb")
dt64 = fin.time.values
dates = pd.to_datetime(dt64)
thres_file="/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/t95_baseline.nc"
bsyear=1979
beyear=2000
month_starty=7
EHFaccl=True
method="NF13"
mask=None
season="yearly"


In [7]:
tave

<xarray.DataArray 'tas' (time: 365, lat: 646, lon: 1082)> Size: 2GB
dask.array<xarray-<this-array>, shape=(365, 646, 1082), dtype=float64, chunksize=(365, 646, 185), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 3kB 2009-07-01T12:00:00 ... 2010-06-30T12:...
  * lat      (lat) float64 5kB -57.97 -57.86 -57.75 -57.64 ... 12.76 12.87 12.98
  * lon      (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
    height   float64 8B ...
    crs      int32 4B ...
Attributes:
    long_name:      Near-Surface Air Temperature
    standard_name:  air_temperature
    units:          K
    cell_methods:   time: point (interval: 1H) time: mean (interval: 1D)
    grid_mapping:   crs

In [8]:
"""Function to calculate Excess Heat Factor (EHF) heatwaves from tave calcualted as (tmax+tmin)/2."""
if mask == None:
    mask = np.ones(tave.shape[1:], int)

# PERFORM SOME CHECKS
## This is explicitly checked to preserve compatibility across versions
if (bsyear == None) or (beyear == None):
    sys.exit(
        "ERROR: you didn't provide base period years to compute_EHF function, please revise"
    )
    
years_all = np.asarray([dates[i].year for i in range(len(dates))])
months_all = np.asarray([dates[i].month for i in range(len(dates))])
days_all = np.asarray([dates[i].day for i in range(len(dates))])

In [9]:
# If using PA13, leap days need to be removed

if method == "PA13":

    dates = dates[((months_all == 2) & (days_all == 29)) == False]
    years = np.asarray([dates[i].year for i in range(len(dates))])
    months = np.asarray([dates[i].month for i in range(len(dates))])
    days = np.asarray([dates[i].day for i in range(len(dates))])

    tave = tave[((months_all == 2) & (days_all == 29)) == False, :, :]

else:

    years = np.asarray([dates[i].year for i in range(len(dates))])
    months = np.asarray([dates[i].month for i in range(len(dates))])
    days = np.asarray([dates[i].day for i in range(len(dates))])

In [10]:
# Specify when the year start
# It is important to define seasons (e.g. Souther Hemisphere, month_starty should be in winter)
new_years = years.copy()
new_years[months < month_starty] -= 1

syear = np.min(years)
eyear = np.max(years)
nyears = eyear - syear
nbyears = beyear - bsyear

shift_pct = np.argmax(new_years == syear)

ndays = tave.shape[0]
nlat = tave.shape[1]
nlon = tave.shape[2]

In [11]:

# Calculate percentiles over the base period
pct = calc_percentile(
    tave[(years >= bsyear) & (years <= beyear), :, :],
    nbyears,
    thres_file,
    method=method,
    nwindow=15,
)

Percentiles are retrieved from the thfile provided


In [12]:
tave_3days = np.zeros(tave.shape, dtype=float)
tave_3days = tave.rolling(time=3).mean().fillna(0)

if EHFaccl == True:
    tave_30days = np.zeros(tave.shape, dtype=float)
    tave_30days = tave.rolling(time=30).mean().fillna(0)



In [13]:
###############################################
###############################################
### CALCULATING EHIsig and EHIaccl (if required)

if method == "PA13":
    EHIsig = np.zeros(tave.shape, dtype=float)
    for t in range(ndays):
        EHIsig[t, :, :] = tave_3days[t, :, :] - pct[(t) % 365, :, :]
else:
    EHIsig = tave_3days - pct

if EHFaccl == True:
    EHIaccl = tave_3days - tave_30days


In [14]:
###############################################
###############################################
## CALCULATING EHF and EHF_Exceed

if EHFaccl == True:
    EHF = np.maximum(1, EHIaccl) * EHIsig
else:
    EHF = EHIsig
EHF = EHF.where(EHF < 0, 0)

EHF_exceed = EHF.where(EHF > 0, 1)

if EHFaccl == True:
    del tave_30days, EHIaccl, EHIsig

In [15]:
###### ZEROING DAYS NOT BELONGING TO SUMMER (SH: NOV,DEC,JAN,FEB,MAR; NH: MAY,JUN,JUL,AUG,SEP)
###### Originally used only in PA13 method
if season == "summer_sh":
    EHF_exceed[(months >= 4) & (months <= 10), :, :] = False
    years[(months >= 4) & (months <= 10)] = -99

    ## For heat wave timing purposes
    shift_start_year = (
        dt.datetime(syear, 11, 0o1) - dt.datetime(syear, 0o7, 0o1)
    ).days

elif season == "summer_nh":
    EHF_exceed[(months >= 10) | (months <= 4), :, :] = False
    years[(months >= 10) | (months <= 4)] = -99
    shift_start_year = 0

elif season == "yearly":
    shift_start_year = 0
else:
    raise ValueError(
        "Season not supported: Choose between summer_sh, summer_nh or yearly"
    )


In [ ]:
EHF = EHF.compute()
EHF_exceed = EHF_exceed.compute()

In [ ]:
EHF.where(EHF > 0, 1)

In [ ]:
# def compute_heatwave_metrics(EHF_exceed_1D, EHF_1D, TMP3D_1D):
#     ndays = EHF_1D.shape[0]
#     spell = calc_spell(EHF_exceed_1D)

#     # Initialize outputs
#     avg = np.full_like(EHF_1D, const.missingval)
#     peak = np.full_like(EHF_1D, const.missingval)
#     tmp_peak = np.full_like(EHF_1D, const.missingval)
#     tmp_avg = np.full_like(EHF_1D, const.missingval)
#     ehf_flag = np.zeros_like(EHF_1D)
    
#     for t in range(ndays):
#         if spell[t] != 0:
#             avg[t] = np.mean(EHF_1D[t : t + spell[t]])
#             peak[t] = np.max(EHF_1D[t : t + spell[t]])
#             tmp_peak[t] = np.max(TMP3D_1D[t : t + spell[t]])                                               
#             tmp_ave[t] = np.mean(TMP3D_1D[t : t + spell[t]])                                               
#             ehf_flag[t : t + spell[t]] = EHF_exceed_1D[t : t + spell[t]]
    
#     return avg, peak, tmp_peak, tmp_avg, ehf_flag

In [ ]:
# heatwave_EHF_avg, heatwave_EHF_peak, heatwave_TMP3D_peak, heatwave_TMP3D_ave, heatwave_EHF = xr.apply_ufunc(
#                                                                                                             compute_heatwave_metrics,
#                                                                                                             EHF_exceed,
#                                                                                                             EHF,
#                                                                                                             tave_3days,
#                                                                                                             input_core_dims=[['time'], ['time'], ['time']],
#                                                                                                             output_core_dims=[['time'], ['time'], ['time'], ['time'], ['time']],
#                                                                                                             vectorize=True,
#                                                                                                             dask='parallelized',
#                                                                                                             output_dtypes=[float, float, float, float, float]
#                                                                                                         )

In [ ]:
# # Defining variables for heat wave and spell calculation
# heatwave_EHF_avg = np.ones(tave.shape, dtype=float) * const.missingval
# heatwave_EHF_peak = np.ones(tave.shape, dtype=float) * const.missingval
# heatwave_TMP3D_peak = np.ones(tave.shape, dtype=float) * const.missingval 
# heatwave_TMP3D_ave = np.ones(tave.shape, dtype=float) * const.missingval 
# heatwave_EHF = np.zeros(tave.shape, dtype=bool)
# spell_all = np.zeros(tave.shape, dtype=int)

# for ilat in range(nlat):
#     for ilon in range(nlon):
#         if mask[ilat, ilon] == 1:
#             spell = calc_spell(EHF_exceed[:, ilat, ilon].values)

#             for t in range(ndays):
#                 if spell[t] != 0:
#                     heatwave_EHF_avg[t, ilat, ilon] = np.mean(
#                         EHF[t : t + spell[t], ilat, ilon]
#                     )
#                     heatwave_EHF_peak[t, ilat, ilon] = np.max(
#                         EHF[t : t + spell[t], ilat, ilon]
#                     )
#                     heatwave_TMP3D_peak[t, ilat, ilon] = np.max(
#                         tave_3days[t : t + spell[t], ilat, ilon]
#                     )                                               
#                     heatwave_TMP3D_ave[t, ilat, ilon] = np.mean(
#                         tave_3days[t : t + spell[t], ilat, ilon]
#                     )                                               
#                     heatwave_EHF[t : t + spell[t], ilat, ilon] = EHF_exceed[
#                         t : t + spell[t], ilat, ilon
#                     ]
                    
            
#             spell_all[:, ilat, ilon] = spell

In [ ]:
# ### PULLING OUT HW CHARACTERISTICS

# heatwave_EHF_avg = np.ma.masked_equal(heatwave_EHF_avg, const.missingval)
# heatwave_EHF_peak = np.ma.masked_equal(heatwave_EHF_peak, const.missingval)

# tave_peak_masked = np.ma.masked_equal(heatwave_TMP3D_peak, const.missingval) 
# tave_avg_masked  = np.ma.masked_equal(heatwave_TMP3D_ave, const.missingval)  


# HWA = np.ones((nyears,) + tave.shape[1:], float) * const.missingval
# HWM = np.ones((nyears,) + tave.shape[1:], float) * const.missingval
# HWN = np.ones((nyears,) + tave.shape[1:], float) * const.missingval
# HWF = np.ones((nyears,) + tave.shape[1:], float) * const.missingval
# HWD = np.ones((nyears,) + tave.shape[1:], float) * const.missingval
# HWT = np.ones((nyears,) + tave.shape[1:], float) * const.missingval
# HWMt = np.ones((nyears,) + tave.shape[1:], float) * const.missingval
# HWAt = np.ones((nyears,) + tave.shape[1:], float) * const.missingval
# HWL = np.ones((nyears,) + tave.shape[1:], float) * const.missingval

# for yr in range(nyears):
#     year = yr + syear
#     HWA[yr, :, :] = np.ma.max(heatwave_EHF_peak[new_years == year, :, :], axis=0)
#     HWM[yr, :, :] = np.ma.mean(heatwave_EHF_avg[new_years == year, :, :], axis=0)
#     HWF[yr, :, :] = (
#         np.sum(spell_all[new_years == year, :, :], axis=0)
#         * 100.0
#         / float(np.sum(new_years == year))
#     )
#     HWN[yr, :, :] = np.sum(spell_all[new_years == year, :, :] != 0, axis=0)
#     HWD[yr, :, :] = np.max(spell_all[new_years == year, :, :], axis=0)
#     HWL[yr, :, :] = (
#         np.ma.sum(spell_all[new_years == year, :, :], axis=0) / HWN[yr, :, :]
#     )
#     HWT[yr, :, :] = np.argmax(spell_all[new_years == year, :, :] != 0, axis=0)

#     HWAt[yr, :, :] = (
#         np.ma.max(tave_peak_masked[new_years == year, :, :], axis=0) - const.tkelvin
#     )
#     HWMt[yr, :, :] = (
#         np.ma.mean(tave_avg_masked[new_years == year, :, :], axis=0) - const.tkelvin
#     )

# HWT = np.ma.masked_equal(HWT, 0.0)
# HWMt[HWMt == 0] = const.missingval
# HWM[HWM == 0] = const.missingval
# HWL[HWN == 0] = const.missingval
# HWA[HWA == 0] = const.missingval
# HWAt[HWAt == 0] = const.missingval

In [ ]:
# np.nonzero(EHF)
EHF[2,470,56]